# Step 1: Environment Setup & Library Installation

# Step 2: Dependencies Ingestion & Model Selection
This cell imports the libraries and initializes your selected models: EasyOCR for text/bounding-box extraction and SpaCy for NLP parsing.

In [11]:
# Cell 2: Import libraries and initialize pre-trained models
import os
import cv2
import easyocr
import spacy
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pypdf import PdfReader

print("--- Step 1: Model Selection & Initialization ---")

# Initialize Pre-trained EasyOCR Engine
try:
    ocr_reader = easyocr.Reader(['en'], gpu=False) # Switch to True if running on an NVIDIA GPU machine
    print("[SUCCESS] Pre-trained EasyOCR Engine loaded perfectly.")
except Exception as e:
    print(f"[ERROR] Failed to initialize EasyOCR: {e}")

# Initialize Pre-trained SpaCy Model for NER
try:
    nlp_model = spacy.load("en_core_web_sm")
    print("[SUCCESS] Pre-trained SpaCy English NLP pipeline loaded perfectly.")
except Exception as e:
    print(f"[ERROR] Failed to load SpaCy model: {e}")

Using CPU. Note: This module is much faster with a GPU.


--- Step 1: Model Selection & Initialization ---
[SUCCESS] Pre-trained EasyOCR Engine loaded perfectly.
[SUCCESS] Pre-trained SpaCy English NLP pipeline loaded perfectly.


# Step 3: Document Ingestion & Directory Configuration

In [27]:
import os

DATASET_DIR = "idp_dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

# Add an 'r' before the strings to fix the Windows backslash syntax error
dataset_samples = {
    "Invoice": r"C:\Users\DEEPIKA SUNIL\Downloads\INV2.jpg",
    "ID Card": r"C:\Users\DEEPIKA SUNIL\Downloads\id2.jpg",
    "Resume": r"C:\Users\DEEPIKA SUNIL\Downloads\resume.jpg"
}

print("--- Step 2: Ingestion Mappings ---")
for doc_type, file_path in dataset_samples.items():
    if os.path.exists(file_path):
        print(f"[READY] Actual file found for {doc_type}: {file_path}")
    else:
        print(f"[⚠️ ACTION REQUIRED] Please place a real file at: {file_path}")

--- Step 2: Ingestion Mappings ---
[READY] Actual file found for Invoice: C:\Users\DEEPIKA SUNIL\Downloads\INV2.jpg
[READY] Actual file found for ID Card: C:\Users\DEEPIKA SUNIL\Downloads\id2.jpg
[READY] Actual file found for Resume: C:\Users\DEEPIKA SUNIL\Downloads\resume.jpg


# Step 4: Image Preprocessing & OCR Extraction Logic

In [17]:
# Cell 4: OpenCV Preprocessing and OCR Extraction
def extract_raw_document_text(file_path):
    """
    Handles Document Preprocessing (OpenCV) and runs the chosen 
    OCR Model dynamically depending on the file format.
    """
    ext = os.path.splitext(file_path)[1].lower()
    text_segments = []
    
    # 1. Route Multi-page PDFs (e.g., Resumes)
    if ext == '.pdf':
        reader = PdfReader(file_path)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text_segments.append(page_text)
        return "\n".join(text_segments)
        
    # 2. Route Images via OpenCV Enhancement + EasyOCR (e.g., Invoices, ID Cards)
    else:
        img = cv2.imread(file_path)
        if img is None:
            return ""
            
        # OpenCV Enhancements
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) # Convert to Grayscale
        blur = cv2.medianBlur(gray, 3)               # Noise reduction
        thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1] # Binarization
        
        # EasyOCR text frame collection
        ocr_results = ocr_reader.readtext(thresh)
        for (bbox, text, confidence) in ocr_results:
            text_segments.append(text)
            
        return " ".join(text_segments)

print("[SUCCESS] Preprocessing & OCR Extraction modules defined.")

[SUCCESS] Preprocessing & OCR Extraction modules defined.


# Step 5: Text Cleaning & Normalization

In [18]:
# Cell 5: Step 5 Data Cleaning Function
def clean_and_normalize_ocr_text(raw_text):
    """
    Cleans raw OCR extractions by removing system noise characters,
    normalizing whitespace fragmentation, and fixing broken formatting.
    """
    if not raw_text:
        return ""
        
    # 1. Standardize formatting markers (Replace system breaks and tabs with clean space)
    cleaned = re.sub(r'[\r\n\t]+', ' ', raw_text)
    
    # 2. Drop common hallucinated OCR noise artifacts
    cleaned = re.sub(r'[\|~`\*\_^]', '', cleaned)
    
    # 3. Collapse multi-space gaps into a single space character
    cleaned = re.sub(r'\s+', ' ', cleaned)
    
    return cleaned.strip()

print("[SUCCESS] Step 5 Data Cleaning framework established.")

[SUCCESS] Step 5 Data Cleaning framework established.


# Step 6: NLP & Hybrid Entity Extraction Logic

In [19]:
# Cell 6: NLP Named Entity Recognition & Regex Engine
def extract_document_entities(cleaned_text, doc_type):
    """
    Combines rule-based regex patterns alongside Pre-trained SpaCy NER
    to map raw text into structured data schemas based on document layout.
    """
    extracted_record = {"Document_Classification": doc_type}
    doc = nlp_model(cleaned_text)
    
    if doc_type == "Invoice":
        # Regex mappings
        inv_match = re.search(r'(?:Invoice|INV|Bill)\s*(?:#|No)?\s*[:.-]?\s*([A-Za-z0-9-]+)', cleaned_text, re.IGNORECASE)
        amt_match = re.search(r'(?:Total|Amount Due|Grand Total)\s*[:.-]?\s*(?:[\$\₹]|USD|INR)?\s*([0-9:,. ]+)', cleaned_text, re.IGNORECASE)
        
        extracted_record["Invoice_Number"] = inv_match.group(1) if inv_match else "Not Found"
        extracted_record["Total_Amount"] = amt_match.group(1).strip() if amt_match else "Not Found"
        
        # Pre-trained SpaCy entities
        orgs = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
        dates = [ent.text for ent in doc.ents if ent.label_ == "DATE"]
        extracted_record["Vendor_Company"] = orgs[0] if orgs else "Not Found"
        extracted_record["Invoice_Date"] = dates[0] if dates else "Not Found"

    elif doc_type == "ID Card":
        # Regex for standard multi-digit or hyphenated ID formats
        id_match = re.search(r'\b\d{4}-\d{4}-\d{4}\b|\b[A-Z]{3}\d{7}\b', cleaned_text)
        dob_match = re.search(r'\b\d{2}/\d{2}/\d{4}\b|\b\d{2}-\d{2}-\d{4}\b', cleaned_text)
        
        extracted_record["Unique_ID"] = id_match.group(0) if id_match else "Not Found"
        extracted_record["DOB"] = dob_match.group(0) if dob_match else "Not Found"
        
        # Pre-trained SpaCy Person Entity
        persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
        extracted_record["Cardholder_Name"] = persons[0] if persons else "Not Found"

    elif doc_type == "Resume":
        # Regex for communication channels
        email_match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', cleaned_text)
        extracted_record["Email_Address"] = email_match.group(0) if email_match else "Not Found"
        
        # Contextual SpaCy Entity Identification
        persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
        extracted_record["Candidate_Name"] = persons[0] if persons else "Not Found"
        
        # Rule-based programmatic parsing for technical skill keywords
        tech_keywords = ["Python", "SQL", "Machine Learning", "NLP", "OpenCV", "Streamlit", "AWS"]
        matched_skills = [skill for skill in tech_keywords if skill.lower() in cleaned_text.lower()]
        extracted_record["Skills_Inventory"] = ", ".join(matched_skills) if matched_skills else "Not Found"

    return extracted_record

print("[SUCCESS] NLP Entity Extraction layers successfully mapped.")

[SUCCESS] NLP Entity Extraction layers successfully mapped.


# Step 7: Post-processing, Validation & Pipeline Loop Execution
The final cell iterates through your data directory, triggers your end-to-end extraction pipeline, validates the results, and outputs a structured Pandas DataFrame.

In [28]:
# Cell 7: End-to-End Pipeline Evaluation Run
all_final_outputs = []

print("--- Initializing Automated End-to-End Evaluation Loop ---\n")

for doc_type, file_path in dataset_samples.items():
    if not os.path.exists(file_path):
        print(f"⚠️ [SKIPPED] {doc_type} sample missing from disk. Place a file at '{file_path}' to test.")
        continue
        
    print(f"⚙️ Running pipeline on actual {doc_type} asset...")
    
    # Step 4: Extract Raw Data
    raw_ocr = extract_raw_document_text(file_path)
    
    # Step 5: Clean and Normalize Text
    cleaned_ocr = clean_and_normalize_ocr_text(raw_ocr)
    
    # Step 6: Information Extraction Pipeline
    structured_dict = extract_document_entities(cleaned_ocr, doc_type)
    
    # Step 7: Post-Processing Data Type Normalization / Cleaning
    if "Total_Amount" in structured_dict and structured_dict["Total_Amount"] != "Not Found":
        # Remove commas or stray symbols to convert string digits to float
        numeric_clean = re.sub(r'[^\d.]', '', structured_dict["Total_Amount"])
        try:
            structured_dict["Total_Amount"] = float(numeric_clean)
        except ValueError:
            pass # Keep original string if formatting fails
            
    all_final_outputs.append(structured_dict)

# Render complete system overview as an analytical tabular matrix
print("\n📊 FINAL IDP STRUCTURAL VALIDATION MATRIX:")
if all_final_outputs:
    df_evaluation_summary = pd.DataFrame(all_final_outputs)
    display(df_evaluation_summary)
else:
    print("No files were detected inside the 'idp_dataset/' folder. Drop test documents in to generate rows.")

--- Initializing Automated End-to-End Evaluation Loop ---

⚙️ Running pipeline on actual Invoice asset...


c:\Users\DEEPIKA SUNIL\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


⚙️ Running pipeline on actual ID Card asset...
⚙️ Running pipeline on actual Resume asset...

📊 FINAL IDP STRUCTURAL VALIDATION MATRIX:


,Document_Classification,Invoice_Number,Total_Amount,Vendor_Company,Invoice_Date,Unique_ID,DOB,Cardholder_Name,Email_Address,Candidate_Name,Skills_Inventory
0,Invoice,oicc,,Invoicc,12345,NaN,NaN,NaN,NaN,NaN,NaN
1,ID Card,NaN,NaN,NaN,NaN,Not Found,Not Found,I15,NaN,NaN,NaN
2,Resume,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not Found,Not Found,Not Found
